In [ ]:
import polars as pl
import requests
import os
from dotenv import load_dotenv
import json

print("Libraries imported successfully.")

```python
API_KEY="your_actual_api_key_here"
BASE_API_URL="https://api.energyid.eu"
EXCEL_FILE_PATH="path/to/your/installations.xlsx"
```

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get configuration from environment variables
API_KEY = os.getenv("API_KEY")
BASE_API_URL = os.getenv("BASE_API_URL")
EXCEL_FILE_PATH = os.getenv("EXCEL_FILE_PATH")

# Prepare headers for API requests
headers = {
    "Authorization": f"apiKey {API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# Load the source CSV file
try:
    df = pl.read_csv(EXCEL_FILE_PATH)
    # Combine Street and Number for the full address
    df = df.with_columns(
        (pl.col("Straat") + " " + pl.col("Nummer").cast(pl.Utf8)).alias(
            "full_street_address"
        )
    )
    print(f"Successfully loaded {len(df)} records from '{EXCEL_FILE_PATH}'.")
    print("\nData preview:")
    print(df.head(3))
except Exception as e:
    print(f"Error loading file '{EXCEL_FILE_PATH}': {e}")
    df = None

In [ ]:
df.describe()

In [ ]:
if df is not None:
    print("\n--- Starting Dossier Creation Process ---")

    # Define the API endpoint URL once
    create_record_url = f"{BASE_API_URL}/api/v1/records"

    for i, row in enumerate(df.iter_rows(named=True)):
        display_name = row.get("Naamgeving installatie Energie ID")
        print(f"\nProcessing {i + 1}/{len(df)}: '{display_name}'...")

        # Construct the payload for the API request from the row data.
        # This dictionary will be converted to URL query parameters.
        params_payload = {
            "displayName": display_name,
            "recordType": "ProductionUnit",
            "category": "Production",
            "streetAddress": row.get("full_street_address"),
            "city": row.get("Plaatsnaam"),
            "postalCode": str(row.get("postcode")),
            "country": "BE",
            "pvPeakPower": row.get("Piekvermogen (kWp)"),
            # The 'tags' parameter will be sent as multiple query keys, which is standard
            "tags": [
                f"Costcenter:{row.get('Costcenter')}"
                if row.get("Costcenter")
                else None,
                "script_created_dossier",  # Tag for easy identification and cleanup
            ],
        }

        # Remove any keys with None values
        params_payload = {k: v for k, v in params_payload.items() if v is not None}
        params_payload["tags"] = [
            tag for tag in params_payload.get("tags", []) if tag is not None
        ]

        # Make the API call to create the record
        try:
            # Use params= instead of json= to send data as URL query parameters
            response = requests.post(
                create_record_url, headers=headers, params=params_payload
            )
            response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

            new_record = response.json()
            record_id = new_record.get("id")
            record_number = new_record.get("recordNumber")
            print(f"  ✅ SUCCESS: Created Dossier '{display_name}'.")
            print(f"     - Record ID: {record_id}")
            print(f"     - Record Number: {record_number}")

        except requests.exceptions.HTTPError as e:
            print(f"  ❌ FAILURE: Could not create dossier for '{display_name}'.")
            print(f"     - Status Code: {e.response.status_code}")
            try:
                error_details = e.response.json()
                print(f"     - API Error: {json.dumps(error_details)}")
            except json.JSONDecodeError:
                print(f"     - Response Body: {e.response.text}")
        except requests.exceptions.RequestException as e:
            print(f"  ❌ FAILURE: A network error occurred for '{display_name}': {e}")

    print("\n--- Dossier Creation Process Finished ---")
else:
    print("DataFrame not loaded. Halting execution.")

In [ ]:
# --- Simple GET request to test API Key and Base URL ---
print("--- Testing API Key and Base URL with a GET request ---")
test_get_url = f"{BASE_API_URL}/api/v1/members/me"
print("Requesting URL:", test_get_url)
print("Using Headers:", headers)
get_response = requests.get(test_get_url, headers=headers)

# Print raw response details
print("\n--- GET Response ---")
print("Response Object:", get_response)
print("Status Code:", get_response.status_code)
print("Response Headers:", get_response.headers)
print("Response Text:", get_response.text)
print("--- End of GET Test ---")


# --- Replicating the failing POST request for the first row for debugging ---
print("\n\n--- Re-running the POST request for the first data row ---")
first_row = df.head(1).to_dicts()[0]

# Construct the payload exactly as before
post_payload = {
    "displayName": first_row.get("Naamgeving installatie Energie ID"),
    "recordType": "ProductionUnit",
    "category": "Production",
    "streetAddress": first_row.get("full_street_address"),
    "city": first_row.get("Plaatsnaam"),
    "postalCode": str(first_row.get("postcode")),
    "country": "BE",
    "pvPeakPower": first_row.get("Piekvermogen (kWp)"),
    "tags": [
        f"Costcenter:{first_row.get('Costcenter')}"
        if first_row.get("Costcenter")
        else None,
        "script_created_dossier",
    ],
}

# Clean up None values
post_payload = {k: v for k, v in post_payload.items() if v is not None}
post_payload["tags"] = [tag for tag in post_payload.get("tags", []) if tag is not None]

# Define the URL for the POST request
test_post_url = f"{BASE_API_URL}/api/v1/records"

print("\nRequesting URL:", test_post_url)
print("Using Headers:", headers)
print("With Payload:", json.dumps(post_payload, indent=2))

# Make the POST request
post_response = requests.post(test_post_url, headers=headers, json=post_payload)

# Print raw response details for the POST request
print("\n--- POST Response ---")
print("Response Object:", post_response)
print("Status Code:", post_response.status_code)
print("Response Headers:", post_response.headers)
print("Response Text:", post_response.text)
print("--- End of POST Test ---")

In [ ]:
def delete_script_created_dossiers(tag_to_identify="script_created_dossier"):
    """
    Finds and deletes all records in the account that have a specific tag.
    """
    print(f"\n--- Searching for dossiers to delete with tag: '{tag_to_identify}' ---")

    try:
        # Get all records for the user
        response = requests.get(
            f"{BASE_API_URL}/api/v1/members/me/records", headers=headers
        )
        response.raise_for_status()
        all_records = response.json()

        # Filter for records with the specific tag
        records_to_delete = [
            record
            for record in all_records
            if tag_to_identify in record.get("tags", [])
        ]

        if not records_to_delete:
            print("No dossiers found with the specified tag. Nothing to do.")
            return

        print(f"Found {len(records_to_delete)} dossiers to delete. Proceeding...")
        deleted_count = 0

        for record in records_to_delete:
            record_id = record.get("id")
            record_name = record.get("displayName")
            print(f"  Attempting to delete '{record_name}' (ID: {record_id})...")

            del_response = requests.delete(
                f"{BASE_API_URL}/api/v1/records/{record_id}", headers=headers
            )

            if del_response.status_code in [200, 204]:
                print("    ✅ DELETED successfully.")
                deleted_count += 1
            else:
                print(
                    f"    ❌ FAILED to delete. Status: {del_response.status_code}, Info: {del_response.text}"
                )

        print(f"\n--- Cleanup Complete: Deleted {deleted_count} dossiers. ---")

    except requests.exceptions.RequestException as e:
        print(f"  ❌ ERROR during cleanup: {e}")


# --- To run the cleanup, uncomment the line below ---
# delete_script_created_dossiers()

In [ ]:
def delete_fi_dossiers():
    """
    Finds and deletes all records that have displayName starting with "FI - "
    (but NOT "FII - " to preserve the correct ones).
    """
    print("\n--- Searching for FI - dossiers to delete ---")

    try:
        # Get all records for the user
        response = requests.get(
            f"{BASE_API_URL}/api/v1/members/me/records", headers=headers
        )
        response.raise_for_status()
        all_records = response.json()

        # Filter for records with displayName starting with "FI - " but NOT "FII - "
        records_to_delete = [
            record
            for record in all_records
            if (
                record.get("displayName", "").startswith("FI - ")
                and not record.get("displayName", "").startswith("FII - ")
            )
        ]

        if not records_to_delete:
            print("No FI - dossiers found. Nothing to delete.")
            return

        print(f"Found {len(records_to_delete)} FI - dossiers to delete:")
        for record in records_to_delete:
            print(f"  - {record.get('displayName')} (ID: {record.get('id')})")

        # Ask for confirmation before proceeding
        print(f"\nThis will delete {len(records_to_delete)} dossiers. Proceeding...")
        deleted_count = 0

        for record in records_to_delete:
            record_id = record.get("id")
            record_name = record.get("displayName")
            print(f"  Deleting '{record_name}' (ID: {record_id})...")

            del_response = requests.delete(
                f"{BASE_API_URL}/api/v1/records/{record_id}", headers=headers
            )

            if del_response.status_code in [200, 204]:
                print("    ✅ DELETED successfully.")
                deleted_count += 1
            else:
                print(
                    f"    ❌ FAILED to delete. Status: {del_response.status_code}, Info: {del_response.text}"
                )

        print(f"\n--- Cleanup Complete: Deleted {deleted_count} FI - dossiers. ---")
        print("FII - dossiers were preserved and not touched.")

    except requests.exceptions.RequestException as e:
        print(f"  ❌ ERROR during cleanup: {e}")


# --- To run the cleanup, uncomment the line below ---
delete_fi_dossiers()

In [ ]:
def to_proper_case(text):
    """
    Convert text to proper case (Title Case) while handling special cases.
    """
    if not text:
        return text

    # Handle special cases for Dutch street names
    special_words = {
        "van": "van",
        "de": "de",
        "der": "der",
        "den": "den",
        "het": "het",
        "op": "op",
        "ter": "ter",
        "ten": "ten",
        "aan": "aan",
        "bij": "bij",
    }

    words = str(text).split()
    result = []

    for i, word in enumerate(words):
        # First word is always capitalized
        if i == 0:
            result.append(word.capitalize())
        # Check if it's a special word (lowercase)
        elif word.lower() in special_words:
            result.append(special_words[word.lower()])
        else:
            result.append(word.capitalize())

    return " ".join(result)


def create_dossiers_with_proper_case(df):
    """
    Create dossiers with proper case formatting for names and addresses.
    """
    if df is not None:
        print("\n--- Starting Dossier Creation Process (with Proper Case) ---")

        # Clean column names first (for Polars)
        print("Original columns:", df.columns)
        cleaned_columns = [col.strip() for col in df.columns]
        df = df.rename(dict(zip(df.columns, cleaned_columns)))
        print("Cleaned columns:", df.columns)

        # Define the API endpoint URL once
        create_record_url = f"{BASE_API_URL}/api/v1/records"

        for i, row in enumerate(df.iter_rows(named=True)):
            # Get the original display name and convert to proper case
            original_display_name = row.get("Naamgeving installatie Energie ID")

            # For display names like "FI - KAREKIETSTRAAT 1", convert the street part
            if original_display_name:
                # Split on the pattern "FI - " or "FII - "
                if original_display_name.startswith("FI - "):
                    prefix = "FI - "
                    street_part = original_display_name[5:]  # Remove "FI - "
                elif original_display_name.startswith("FII - "):
                    prefix = "FII - "
                    street_part = original_display_name[6:]  # Remove "FII - "
                else:
                    prefix = ""
                    street_part = original_display_name

                # Convert street part to proper case
                proper_street = to_proper_case(street_part)
                display_name = prefix + proper_street
            else:
                display_name = original_display_name

            print(f"\nProcessing {i + 1}/{len(df)}: '{display_name}'...")

            # Get street address and convert to proper case
            street_address = row.get("full_street_address")
            if street_address:
                street_address = to_proper_case(street_address)

            # Get city and convert to proper case
            city = row.get("Plaatsnaam")
            if city:
                city = to_proper_case(city)

            # Construct the payload for the API request
            params_payload = {
                "displayName": display_name,
                "recordType": "ProductionUnit",
                "category": "Production",
                "streetAddress": street_address,
                "city": city,
                "postalCode": str(row.get("postcode")) if row.get("postcode") else None,
                "country": "BE",
                "pvPeakPower": row.get("Piekvermogen [kWp]"),
                "tags": [
                    f"Costcenter:{row.get('Costcenter')}"
                    if row.get("Costcenter")
                    else None,
                    "script_created_dossier",  # Tag for easy identification
                ],
            }

            # Remove any keys with None values
            params_payload = {k: v for k, v in params_payload.items() if v is not None}
            params_payload["tags"] = [
                tag for tag in params_payload.get("tags", []) if tag is not None
            ]

            # Make the API call to create the record
            try:
                response = requests.post(
                    create_record_url, headers=headers, params=params_payload
                )
                response.raise_for_status()

                new_record = response.json()
                record_id = new_record.get("id")
                record_number = new_record.get("recordNumber")
                print(f"  ✅ SUCCESS: Created Dossier '{display_name}'.")
                print(f"     - Record ID: {record_id}")
                print(f"     - Record Number: {record_number}")

            except requests.exceptions.HTTPError as e:
                print(f"  ❌ FAILURE: Could not create dossier for '{display_name}'.")
                print(f"     - Status Code: {e.response.status_code}")
                try:
                    error_details = e.response.json()
                    print(f"     - API Error: {json.dumps(error_details)}")
                except json.JSONDecodeError:
                    print(f"     - Response Body: {e.response.text}")
            except requests.exceptions.RequestException as e:
                print(
                    f"  ❌ FAILURE: A network error occurred for '{display_name}': {e}"
                )

        print("\n--- Dossier Creation Process Finished ---")
    else:
        print("DataFrame not loaded. Halting execution.")


# Example usage:
# 1. First delete the incorrect FI dossiers:
# delete_fi_dossiers()

# 2. Then create new ones with proper case:
create_dossiers_with_proper_case(df)

In [ ]:
import requests
import json


def delete_empty_grouped_dossiers(dry_run=True):
    """
    Find and delete empty grouped dossiers (like "Otterbeek - Karekietstraat 1 - 7").
    These are typically the range-grouped dossiers that might be empty.

    Args:
        dry_run (bool): If True, only shows what would be deleted without actually deleting
    """
    print("\n--- Searching for Empty Grouped Dossiers ---")

    try:
        # Get all records for the current user
        response = requests.get(
            f"{BASE_API_URL}/api/v1/members/me/records", headers=headers
        )
        response.raise_for_status()
        all_records = response.json()

        print(f"Found {len(all_records)} total records in your account.")

        # Filter for grouped dossiers (those with ranges like "1 - 7", "10 - 16", etc.)
        grouped_pattern_keywords = [
            "Otterbeek - Karekietstraat",
            "Otterbeek - Tivolilaan",
            " - ",  # Contains a range pattern
        ]

        # First, identify potential grouped dossiers
        potential_grouped = []
        for record in all_records:
            display_name = record.get("displayName", "")
            # Check if it contains range patterns like "1 - 7" or location patterns
            if any(keyword in display_name for keyword in grouped_pattern_keywords):
                # Check if it has range numbers (e.g., "1 - 7", "10 - 16")
                if " - " in display_name and any(
                    c.isdigit() for c in display_name.split(" - ")[-1]
                ):
                    potential_grouped.append(record)

        print(f"Found {len(potential_grouped)} potential grouped dossiers:")
        for record in potential_grouped:
            print(f"  - {record.get('displayName')} (ID: {record.get('id')})")

        if not potential_grouped:
            print("No grouped dossiers found to check.")
            return

        # Now check each one to see if it's empty
        empty_dossiers = []

        for record in potential_grouped:
            record_id = record.get("id")
            record_name = record.get("displayName")

            print(f"\nChecking '{record_name}'...")

            # Check if the record has any meters
            meters_response = requests.get(
                f"{BASE_API_URL}/api/v1/records/{record_id}/meters", headers=headers
            )

            if meters_response.status_code == 200:
                meters = meters_response.json()
                meter_count = len(meters) if meters else 0

                # Check for other indicators of usage
                has_recent_activity = record.get("lastSubmissionAt") is not None

                print(f"  - Meters: {meter_count}")
                print(f"  - Last submission: {record.get('lastSubmissionAt', 'None')}")

                # Consider it empty if it has no meters and no recent activity
                if meter_count == 0 and not has_recent_activity:
                    empty_dossiers.append(record)
                    print("  ✓ EMPTY - candidate for deletion")
                else:
                    print("  ✗ HAS DATA - keeping")
            else:
                print(f"  ❌ Error checking meters: {meters_response.status_code}")

        if not empty_dossiers:
            print(
                f"\n✅ No empty grouped dossiers found. All {len(potential_grouped)} grouped dossiers contain data."
            )
            return

        print(f"\n📋 Summary: Found {len(empty_dossiers)} empty grouped dossiers:")
        for record in empty_dossiers:
            print(f"  - {record.get('displayName')} (ID: {record.get('id')})")

        if dry_run:
            print("\n🔍 DRY RUN MODE: No dossiers were deleted.")
            print(
                f"To actually delete these {len(empty_dossiers)} empty dossiers, run:"
            )
            print("delete_empty_grouped_dossiers(dry_run=False)")
            return

        # Actually delete the empty dossiers
        print(f"\n🗑️  DELETING {len(empty_dossiers)} empty grouped dossiers...")
        deleted_count = 0

        for record in empty_dossiers:
            record_id = record.get("id")
            record_name = record.get("displayName")

            print(f"  Deleting '{record_name}'...")
            del_response = requests.delete(
                f"{BASE_API_URL}/api/v1/records/{record_id}", headers=headers
            )

            if del_response.status_code in [200, 204]:
                print("    ✅ DELETED successfully.")
                deleted_count += 1
            else:
                print(f"    ❌ FAILED to delete. Status: {del_response.status_code}")
                try:
                    error_details = del_response.json()
                    print(f"       Error: {json.dumps(error_details)}")
                except Exception:
                    print(f"       Response: {del_response.text}")

        print(
            f"\n--- Cleanup Complete: Deleted {deleted_count} empty grouped dossiers ---"
        )

    except requests.exceptions.RequestException as e:
        print(f"  ❌ ERROR during cleanup: {e}")


# Alternative: Delete specific dossiers by pattern
def delete_dossiers_by_pattern(pattern, dry_run=True):
    """
    Delete dossiers that match a specific pattern in their display name.

    Args:
        pattern (str): Pattern to match in displayName (e.g., "Otterbeek - Karekietstraat")
        dry_run (bool): If True, only shows what would be deleted
    """
    print(f"\n--- Searching for dossiers matching pattern: '{pattern}' ---")

    try:
        response = requests.get(
            f"{BASE_API_URL}/api/v1/members/me/records", headers=headers
        )
        response.raise_for_status()
        all_records = response.json()

        # Filter for records matching the pattern
        matching_records = [
            record
            for record in all_records
            if pattern.lower() in record.get("displayName", "").lower()
        ]

        if not matching_records:
            print(f"No dossiers found matching pattern '{pattern}'.")
            return

        print(f"Found {len(matching_records)} dossiers matching pattern:")
        for record in matching_records:
            print(f"  - {record.get('displayName')} (ID: {record.get('id')})")

        if dry_run:
            print("\n🔍 DRY RUN MODE: No dossiers were deleted.")
            print(f"To actually delete these {len(matching_records)} dossiers, run:")
            print(f"delete_dossiers_by_pattern('{pattern}', dry_run=False)")
            return

        # Delete the matching dossiers
        print(f"\n🗑️  DELETING {len(matching_records)} matching dossiers...")
        deleted_count = 0

        for record in matching_records:
            record_id = record.get("id")
            record_name = record.get("displayName")

            print(f"  Deleting '{record_name}'...")
            del_response = requests.delete(
                f"{BASE_API_URL}/api/v1/records/{record_id}", headers=headers
            )

            if del_response.status_code in [200, 204]:
                print("    ✅ DELETED successfully.")
                deleted_count += 1
            else:
                print(f"    ❌ FAILED to delete. Status: {del_response.status_code}")

        print(f"\n--- Pattern Cleanup Complete: Deleted {deleted_count} dossiers ---")

    except requests.exceptions.RequestException as e:
        print(f"  ❌ ERROR during cleanup: {e}")


# --- Usage Examples ---

# 1. Check for empty grouped dossiers (DRY RUN - safe to test)
delete_empty_grouped_dossiers(dry_run=True)

# 2. Actually delete empty grouped dossiers (BE CAREFUL!)
# delete_empty_grouped_dossiers(dry_run=False)

# 3. Delete all dossiers matching a specific pattern (DRY RUN)
# delete_dossiers_by_pattern("Otterbeek - Karekietstraat", dry_run=True)

# 4. Actually delete by pattern (BE CAREFUL!)
# delete_dossiers_by_pattern("Otterbeek - Karekietstraat", dry_run=False)

# now we do the last stuff for the f2 dossiers